# Movie Recommender — Sentence Embedding Approach

**Why I switched from TF-IDF:**  
TF-IDF counts word frequencies — it has no concept of *meaning* or *vibe*.  
Two movies sharing an actor or the word 'time' score as similar even if they feel completely different.

**What sentence embeddings do differently:**  
A neural network converts each movie description into a vector of numbers that captures *semantic meaning*.  
"A bittersweet romance across time" and "a magical love story with emotional depth" end up close together  
in that vector space — even though they share no words. That's exactly what we need.

**Model used:** `all-MiniLM-L6-v2` — small (~90MB), fast, and excellent for semantic similarity.

## Step 0: Install Dependencies

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('All imports OK')

c:\Users\User\Desktop\Visual studio\movie-rec\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


## Step 1: Load & Merge the Data

In [2]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits, on='title')

print(f'Loaded {len(movies)} movies.')

# Keep only the columns we need
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(subset=['overview'], inplace=True)

print(f'After dropping missing overviews: {len(movies)} movies.')

Loaded 4809 movies.
After dropping missing overviews: 4806 movies.


## Step 2: Parse JSON Columns

The genres/keywords/cast/crew columns are stored as JSON strings.  
We convert them into plain Python lists.

In [3]:
def extract_names(text):
    """Converts '[{"name": "Action"}, ...]' → ['Action', ...]"""
    try:
        return [item['name'] for item in ast.literal_eval(text)]
    except:
        return []

def extract_top_cast(text, n=1):
    """
    Returns only the lead actor (n=1).
    Using 3 cast members caused wrong recommendations — e.g. Inception → 500 Days of Summer
    just because Joseph Gordon-Levitt is in both. One lead actor is enough signal.
    """
    try:
        return [item['name'] for item in ast.literal_eval(text)[:n]]
    except:
        return []

def extract_director(text):
    """Finds the person whose job is 'Director' in the crew list."""
    try:
        for item in ast.literal_eval(text):
            if item['job'] == 'Director':
                return [item['name']]
        return []
    except:
        return []

movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(extract_top_cast)
movies['crew']     = movies['crew'].apply(extract_director)

print('Parsed. Sample genres for Avatar:', movies[movies['title'] == 'Avatar']['genres'].values[0])

Parsed. Sample genres for Avatar: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']


## Step 3: Build Rich Natural Language Tags

**Key difference from the old approach:**  
Before, we built a jumble of collapsed words: `"timetravel lovestory domhnallgleeson..."`  
Now we build proper sentences: `"A young man who can time travel falls in love. This is a Romance, Drama film. Themes include time travel, love, fate."`

The embedding model understands sentences far better than word soup.

In [4]:
def build_rich_tags(row):
    """
    Builds a human-readable description for each movie.
    Each part adds semantic context the embedding model can understand.
    """
    parts = []

    # The overview is the most important — it describes the actual story
    if row['overview']:
        parts.append(row['overview'])

    # Genres as a natural sentence
    if row['genres']:
        parts.append(f"This is a {', '.join(row['genres'])} film.")

    # Top 8 keywords as themes
    if row['keywords']:
        parts.append(f"Themes include {', '.join(row['keywords'][:8])}.")

    # Director
    if row['crew']:
        parts.append(f"Directed by {row['crew'][0]}.")

    # Lead actor only
    if row['cast']:
        parts.append(f"Starring {row['cast'][0]}.")

    return ' '.join(parts)


movies['tags'] = movies.apply(build_rich_tags, axis=1)

# Build the final dataframe — keep genres for the scoring step later
final = movies[['movie_id', 'title', 'tags', 'genres']].reset_index(drop=True)

print("Sample tag for 'About Time':")
about_time = final[final['title'] == 'About Time']
if not about_time.empty:
    print(about_time['tags'].values[0])
else:
    print('(About Time not in dataset — showing Avatar instead)')
    print(final[final['title'] == 'Avatar']['tags'].values[0])

Sample tag for 'About Time':
The night after another unsatisfactory New Year party, Tim's father tells his son that the men in his family have always had the ability to travel through time. Tim can't change history, but he can change what happens and has happened in his own life – so he decides to make his world a better place... by getting a girlfriend. Sadly, that turns out not to be as easy as he thinks. This is a Comedy, Drama, Science Fiction film. Themes include london england, father son relationship, time travel. Directed by Richard Curtis. Starring Domhnall Gleeson.


## Step 4: Generate Sentence Embeddings

This is where the magic happens.  
The model converts each movie's tag string into a vector of 384 numbers.  
Movies with similar *meaning* end up with similar vectors.

**First run:** downloads the model (~90MB). Subsequent runs use the cached version.  
**Time:** about 1–2 minutes for ~4800 movies.

In [5]:
# Load the model (downloads on first run, cached after)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Generating embeddings for {len(final)} movies...')
print('This takes 1-2 minutes — progress bar below:')

embeddings = model.encode(
    final['tags'].tolist(),
    show_progress_bar=True,
    batch_size=64,          # process 64 movies at a time
)

print(f'\nEmbeddings shape: {embeddings.shape}')
# Should be (4806, 384) — 4806 movies, 384 dimensions per movie

c:\Users\User\Desktop\Visual studio\movie-rec\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5191.03it/s]


Generating embeddings for 4806 movies...
This takes 1-2 minutes — progress bar below:


Batches: 100%|██████████| 76/76 [02:46<00:00,  2.19s/it]


Embeddings shape: (4806, 384)


## Step 5: Compute Cosine Similarity

Same math as before — measures the angle between two vectors.  
Score of 1.0 = identical meaning, 0.0 = completely unrelated.  
The difference is the *input* is now semantically rich, not just word frequencies.

In [6]:
similarity = cosine_similarity(embeddings)

print(f'Similarity matrix shape: {similarity.shape}')
# Should be (4806, 4806) — every movie vs every other movie

Similarity matrix shape: (4806, 4806)


## Step 6: Genre Scoring Helpers

Two extra filters on top of the embedding similarity:

- **`genre_overlap_score`** (Fix 4): multiplies the similarity score by a genre bonus.  
  Same genres → full score. No genre overlap → 30% penalty. Keeps scores gradual.

- **`genres_are_compatible`** (Fix 2): a hard filter that blocks horror from appearing  
  in romance results and vice versa. Acts as a safety net after scoring.

In [7]:
# Genre groups that should never cross-recommend
INCOMPATIBLE_GENRES = [
    {'Horror', 'Thriller'},
    {'Romance', 'Comedy', 'Family'},
    {'Animation', 'Family'},
]

def genres_are_compatible(genres_a, genres_b):
    """
    Returns False if one movie belongs to an incompatible genre group and the other doesn't.
    Example: Romance movie vs Horror movie → False (block it)
    Example: Action movie vs Thriller movie → True (allow it)
    """
    set_a = set(genres_a)
    set_b = set(genres_b)
    for group in INCOMPATIBLE_GENRES:
        a_in_group = bool(set_a & group)
        b_in_group = bool(set_b & group)
        if a_in_group != b_in_group:
            return False
    return True


def genre_overlap_score(genres_a, genres_b):
    """
    Jaccard similarity between two genre sets, scaled to 0.3–1.0.
    This multiplies the raw cosine similarity score to boost same-genre matches.
    0.3 floor means even zero-overlap movies still appear if semantically similar enough.
    """
    if not genres_a or not genres_b:
        return 0.7  # neutral if genre data is missing
    set_a = set(genres_a)
    set_b = set(genres_b)
    overlap = len(set_a & set_b)
    union   = len(set_a | set_b)
    jaccard = overlap / union
    return 0.3 + (0.7 * jaccard)


print('Genre helpers defined.')

Genre helpers defined.


## Step 7: The Recommendation Function

Flow:
1. Find the movie in the dataframe
2. Get its embedding similarity scores against all other movies
3. Multiply each score by the genre overlap bonus (Fix 4)
4. Sort by adjusted score
5. Walk down the list, skipping genre-incompatible movies (Fix 2)
6. Return top N

In [8]:
def recommend(movie_title, num_recommendations=10):
    matches = final[final['title'].str.lower() == movie_title.lower()]

    if matches.empty:
        print(f"❌ '{movie_title}' not found!")
        # Suggest partial matches
        close = final[final['title'].str.lower().str.contains(movie_title.lower())]
        if not close.empty:
            print('Did you mean one of these?')
            for t in close['title'].head(5):
                print(f'  - {t}')
        return []

    idx          = matches.index[0]
    query_genres = final.iloc[idx]['genres']

    # Fix 4: adjust raw similarity scores by genre overlap
    sim_scores = []
    for i, raw_score in enumerate(similarity[idx]):
        candidate_genres = final.iloc[i]['genres']
        bonus            = genre_overlap_score(query_genres, candidate_genres)
        adjusted_score   = raw_score * bonus
        sim_scores.append((i, adjusted_score))

    # Sort highest first, skip the movie itself (always rank 0)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]

    # Fix 2: filter out genre-incompatible movies
    results = []
    for movie_idx, score in sim_scores:
        candidate_genres = final.iloc[movie_idx]['genres']
        if genres_are_compatible(query_genres, candidate_genres):
            results.append(final['title'].iloc[movie_idx])
        if len(results) >= num_recommendations:
            break

    return results


print('recommend() defined.')

recommend() defined.


## Step 8: Test It

In [20]:
test_movies = ['About Time', "Love actually", 'Inception', 'Toy Story']

for movie in test_movies:
    print(f"\n🎬 Movies similar to '{movie}':")
    results = recommend(movie)
    if results:
        for i, rec in enumerate(results, 1):
            print(f'  {i}. {rec}')
    else:
        print('  (no results)')


🎬 Movies similar to 'About Time':
  1. Safety Not Guaranteed
  2. Hot Tub Time Machine
  3. Birdman
  4. Driving Lessons
  5. Igby Goes Down
  6. The Way Way Back
  7. Saint Ralph
  8. The Visitors
  9. Tumbleweeds
  10. This Is Where I Leave You

🎬 Movies similar to 'Love actually':
  1. The Family Stone
  2. The Incredibly True Adventure of Two Girls In Love
  3. Raising Helen
  4. When Harry Met Sally...
  5. Hope Springs
  6. Notting Hill
  7. Secretary
  8. Lovely & Amazing
  9. Trust the Man
  10. Bridget Jones's Diary

🎬 Movies similar to 'Inception':
  1. Minority Report
  2. The Maze Runner
  3. Unknown
  4. Paycheck
  5. The Thirteenth Floor
  6. A Sound of Thunder
  7. Cube
  8. Switchback
  9. I Am Number Four
  10. Fortress

🎬 Movies similar to 'Toy Story':
  1. Toy Story 2
  2. Toy Story 3
  3. Free Birds
  4. Deck the Halls
  5. Barnyard
  6. The Boxtrolls
  7. Over the Hedge
  8. Recess: School's Out
  9. Frankenweenie
  10. Pinocchio


## Step 9: Save Outputs

These two files are what Django loads:
- `similarity.pkl` — the full similarity matrix
- `movies_clean.csv` — the cleaned movie dataframe with tags and genres

After saving, copy them to `movie_recommender_backend/ml_data/` and restart Django.

In [ ]:
with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

final.to_csv('movies_clean.csv', index=False)

print('✅ Saved similarity.pkl and movies_clean.csv')
print(f'   {len(final)} movies saved.')
print()
print('Next steps:')
print('  1. cp movies_clean.csv movie_recommender_backend/ml_data/')
print('  2. cp similarity.pkl   movie_recommender_backend/ml_data/')
print('  3. Restart Django (Ctrl+C then python manage.py runserver)')